In [1]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ langchain-community  — installed (0.4.2)
  ✓ lxml  — installed (6.1.1)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Recursive Chunking

Recursive chunking uses `RecursiveCharacterTextSplitter`, which tries an ordered list of separators — `["\n\n", "\n", " ", ""]` by default — and splits on the **largest** separator that keeps each chunk under `chunk_size`. This respects document structure (paragraphs → lines → words) and avoids the oversized-chunk warnings you get from a single fixed separator.

Below we first **load** a PDF and an HTML page, then apply recursive chunking.

## 1. Load source documents

### PDF

Load the sample PDF with `PyPDFLoader` (one `Document` per page).

In [8]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"
pdf_docs = PyPDFLoader(str(pdf_path)).load()

print(f"PDF: loaded {len(pdf_docs)} page(s)")
print(pdf_docs[0].page_content[:300])

PDF: loaded 12 page(s)
SDLC — End-to-End Reference
Page 1
 Software Development Life Cycle
 End-to-End Artifacts & Deliverables
From Business Requirements (BRD) to Release Notes and Operations
 A reference guide describing each SDLC document, its purpose, owner, inputs,
 key contents, and how it feeds the next stage.
 Aut


### HTML

Load the sample HTML with `BSHTMLLoader` (cleaned text in one `Document`).

In [9]:
from langchain_community.document_loaders import BSHTMLLoader

html_path = ROOT / "assets/RAG_Courses.html"
html_docs = BSHTMLLoader(str(html_path)).load()

print(f"HTML: loaded {len(html_docs)} document(s)")
print(html_docs[0].page_content[:300])

HTML: loaded 1 document(s)









12 Best Retrieval-Augmented Generation (RAG) Courses in 2026 — Class Central



















































The Four-Year Degree Isn't Dying — It's Evolving



			View
			




		Close
	









The Report by Class Central


Your source for the latest news and trends in 


## 2. Recursive chunking

### `split_text` vs `split_documents`

Both apply the **same** splitting logic; they differ only in input/output:

| Method | Input | Output | Metadata |
|--------|-------|--------|----------|
| `split_text(text: str)` | a raw string | `list[str]` (plain strings) | **lost** — no `Document` wrapper |
| `split_documents(docs: list[Document])` | `Document` objects (from a loader) | `list[Document]` (chunks) | **preserved** — each source's `.metadata` (page number, source path, …) is copied onto every chunk |

For RAG you usually want **`split_documents`**, so each chunk stays traceable to its origin document.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=500,      # characters per chunk
    chunk_overlap=50,    # characters shared between consecutive chunks
)

# Pass a raw string -> `chunks` is a list[str] with no metadata.
chunks = splitter.split_text(html_docs[0].page_content)
print(f"split_text: {len(chunks)} chunks; first chunk is {len(chunks[0])} characters")
print([len(chunk) for chunk in chunks])

split_text: 101 chunks; first chunk is 475 characters
[475, 487, 472, 378, 461, 370, 177, 479, 481, 432, 81, 45, 483, 499, 61, 483, 43, 377, 316, 450, 250, 278, 286, 486, 486, 16, 291, 421, 251, 489, 255, 336, 329, 495, 282, 226, 347, 469, 495, 304, 483, 300, 309, 326, 491, 262, 356, 233, 470, 125, 316, 280, 425, 479, 328, 409, 412, 445, 68, 255, 329, 392, 475, 69, 287, 450, 367, 420, 161, 298, 437, 476, 444, 118, 266, 380, 387, 427, 73, 276, 302, 386, 464, 128, 233, 423, 390, 463, 485, 454, 479, 464, 474, 458, 464, 483, 480, 464, 474, 494, 39]


In [18]:
# Pass Document objects -> chunks are Documents and keep their source .metadata.
docs = pdf_docs + html_docs
doc_chunks = splitter.split_documents(docs)

print(f"split_documents: {len(docs)} documents -> {len(doc_chunks)} chunks")
print("First chunk metadata:", doc_chunks[0].metadata)
print(f"First chunk ({len(doc_chunks[0].page_content)} chars):\n{doc_chunks[0].page_content[:300]}")
print("last chunk metadata:", doc_chunks[-1].metadata)

print(doc_chunks[0])

split_documents: 13 documents -> 148 chunks
First chunk metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-30T21:30:41-07:00', 'author': 'Prabhukumar Sivamoorthy (Prabhukumarsivamoorthy@gmail.com)', 'keywords': '', 'moddate': '2026-05-30T21:30:41-07:00', 'subject': '(unspecified)', 'title': 'SDLC End-to-End Reference', 'trapped': '/False', 'source': '/Users/Prabhukumar/Projects/PycharmProjects/rag-reference/assets/sample-docs/sdlc-end-to-end.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}
First chunk (361 chars):
SDLC — End-to-End Reference
Page 1
 Software Development Life Cycle
 End-to-End Artifacts & Deliverables
From Business Requirements (BRD) to Release Notes and Operations
 A reference guide describing each SDLC document, its purpose, owner, inputs,
 key contents, and how it feeds the next stage.
 Aut
last chunk metadata: {'source': '/Users/Prabhukumar/Projects/PycharmProjects/rag-reference/assets/RAG_Course